# Predictive Analysis of the Corrected Final Dataset

This notebook implements the predictive workflow after removing the dropped interaction term from the candidate pool.

In [ ]:
from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

DATA_PATH = Path.cwd().parent / 'data' / 'final' / 'final.corrected.csv'
df = pd.read_csv(DATA_PATH)
TARGET = 'log_box_office'
CANDIDATE_PREDICTORS = [
    'audienceScore',
    'tomatoMeter',
    'runtimeMinutes',
    'initial_top_critic_review_count',
    'initial_positive_review_ratio',
    'initial_combined_sentiment_score',
    'log_initial_review_count',
]
INFERENTIAL_MODEL_PREDICTORS = ['audienceScore', 'initial_combined_sentiment_score', 'log_initial_review_count']
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5

for col in [TARGET, 'box_office_num', *CANDIDATE_PREDICTORS]:
    df[col] = pd.to_numeric(df[col], errors='coerce')
predictive_df = df[['id', 'title', 'box_office_num', TARGET, *CANDIDATE_PREDICTORS]].dropna().copy()
predictive_df = predictive_df[np.isfinite(predictive_df[TARGET])].copy()
print(f'Data path: {DATA_PATH}')
print(f'Predictive sample: {len(predictive_df):,}')


## Train/Test Split and Evaluation Helpers

In [ ]:
X = predictive_df[CANDIDATE_PREDICTORS].copy()
y = predictive_df[TARGET].copy()
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(X, y, predictive_df.index, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_df = predictive_df.loc[train_idx].copy()
test_df = predictive_df.loc[test_idx].copy()
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def evaluate_predictions(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

def cv_rmse_for_columns(columns):
    scores = cross_val_score(LinearRegression(), X_train[columns], y_train, cv=cv, scoring='neg_root_mean_squared_error')
    return float(-scores.mean())

def fit_statsmodels_ols(frame, columns):
    return sm.OLS(frame[TARGET], sm.add_constant(frame[columns], has_constant='add')).fit()

def evaluate_linear_model(name, columns):
    fitted = fit_statsmodels_ols(train_df, columns)
    train_pred = fitted.predict(sm.add_constant(train_df[columns], has_constant='add'))
    test_pred = fitted.predict(sm.add_constant(test_df[columns], has_constant='add'))
    train_metrics = evaluate_predictions(train_df[TARGET], train_pred)
    test_metrics = evaluate_predictions(test_df[TARGET], test_pred)
    return {
        'model': name,
        'n_features': len(columns),
        'features': columns,
        'adj_r_squared': float(fitted.rsquared_adj),
        'aic': float(fitted.aic),
        'bic': float(fitted.bic),
        'cv_rmse': cv_rmse_for_columns(columns),
        'test_rmse': test_metrics['rmse'],
        'test_mae': test_metrics['mae'],
        'test_r2': test_metrics['r2'],
    }


## Model Comparison

In [ ]:
results = []
results.append(evaluate_linear_model('Model 1: Inferential three-variable regression', INFERENTIAL_MODEL_PREDICTORS))
results.append(evaluate_linear_model('Model 2: Full candidate-pool regression', CANDIDATE_PREDICTORS))

subset_records = []
for size in range(1, len(CANDIDATE_PREDICTORS) + 1):
    for subset in itertools.combinations(CANDIDATE_PREDICTORS, size):
        fit = fit_statsmodels_ols(train_df, list(subset))
        subset_records.append({
            'n_features': size,
            'features': list(subset),
            'adj_r_squared': float(fit.rsquared_adj),
            'aic': float(fit.aic),
            'bic': float(fit.bic),
            'cv_rmse': cv_rmse_for_columns(list(subset)),
        })

best_subset_table = pd.DataFrame(subset_records)
best_subset_choice = best_subset_table.sort_values(['cv_rmse', 'bic', 'n_features']).iloc[0]
forward_selected = ['log_initial_review_count', 'tomatoMeter', 'audienceScore', 'initial_top_critic_review_count', 'initial_positive_review_ratio']

results.append(evaluate_linear_model('Model 3: Best-subset selected regression', list(best_subset_choice['features'])))
results.append(evaluate_linear_model('Model 4: Forward-stepwise selected regression', forward_selected))

pd.DataFrame(results).round(4)